# Get BDB2026 Player-Tracking Data for DBs

For all of our 2023 rookie DBs, we want to get tracking data for their player plays from the 2023 season using BDB2026 data. 

A very important piece of what we're doing here is ***determining who the WR is that they are guarding***. 

Since we want to find separation during man coverage situations, we will want to find who the WR they're guarding is. Currently, the BDB2026 data has tracking data for all defenders on a given player. We want to determine which WR our player is guarding.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    BDB_2026_BEFORE_PASS_DIR,
    BDB_2026_SUPPLEMENTARY_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

<br><br/>
#### Players Data (*BDB 2027*)

In [ ]:
# =============================
# Players Data
# =============================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

# -----------------------------
# Only Keep Position of Interest
# -----------------------------

players_df = (
    players_df
    .filter(pl.col("nfl_position").is_in(["CB", "FS", "SS"]))
)

# -----------------------------
# Only Keep 2023 Rookies 
# -----------------------------

# We're only keeping 2023 rookies because the 2026 BDB
# player-tracking data only includes 2023 REG season

players_df = (
    players_df
    .filter(
        pl.col("draft_year") == 2023
    )
)

# -----------------------------
# Only Keep Needed Variables
# -----------------------------
players_df = (
    players_df
    .select(
        "nfl_id", 
        "display_name", 
        "draft_year", 
        "nfl_position"
    )
)

# -----------------------------
# Identify Players of Interest
# -----------------------------

players_of_interest = (
    players_df
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()    
)

print("Players Data:")
print(f"    - Number of Players From 2023 Rookie Class: {len(players_of_interest)}")
players_df

<br><br/>
#### Player-Play Data (*BDB 2027*)

In [ ]:
# =============================
# Player-Play Data
# =============================

player_play_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["combine_tracking"]
    )
)

# -----------------------------
# Add game_play_id & player_play_id
# -----------------------------

player_play_df = (
    player_play_df
    .with_columns(
        game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String), 
        player_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String) + "_" + pl.col("nfl_id").cast(pl.String)
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

player_play_df = (
    player_play_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

# ----------------------------
# Only Keep Helpful Variables
# ----------------------------

player_play_df = (
    player_play_df
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "game_play_id", 
        "player_play_id",
    )
)

print("Player-Play Data:")
print(f"    - Number of Players In This Dataset: {player_play_df.select(pl.col("nfl_id")).n_unique()}")
player_play_df

<br><br/>
#### Before Pass Data (*BDB 2026*)

In [ ]:
# ====================================
# Load Before Pass Data
# ====================================

data_dir = Path(str(BDB_2026_BEFORE_PASS_DIR))

# ------------------------------------
# Load and Combine 18 Before Pass Datasets Into 1
# ------------------------------------

before_pass_df = (
    pl.concat(
        [
            pl.scan_csv(data_dir / f"input_2023_w{i:02d}.csv")
            .with_columns(pl.lit(i).alias("week"))
            for i in range(1, 19)
        ],
        how="vertical"
    )
    .collect()
)

# -----------------------------------
# Only Keep DBs & Possible Receivers
# -----------------------------------

before_pass_df = (
    before_pass_df
    .filter(
        (pl.col("nfl_id").is_in(players_of_interest))
        |
        (pl.col("player_position").is_in(["WR"]))
    )
)

# ------------------------------------
# Add Game-Play and Player-Play ID
# ------------------------------------

before_pass_df = (
    before_pass_df
    .with_columns(
        # ID For Play Within A Certain Game
        game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String), 
        # ID For Player-Play Within A Certain Game
        player_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String) + "_" + pl.col("nfl_id").cast(pl.String)
    )
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "frame_id", 
        "game_play_id", 
        "player_play_id", 
        "play_direction",
        "absolute_yardline_number", 
        "player_name", 
        "player_position", 
        "x", 
        "y", 
        "s", 
        "a", 
        "dir", 
        "o"
    )
)

print(f"BDB2026 Player-Tracking Data For Before-Pass Time:")
print(f"    - Positions In Dataset: {before_pass_df.select("player_position").unique().to_series().to_list()}")
before_pass_df

<br><br/>
#### Identify WR Routes That We Have Sufficient Data For

These WR routes can be used to examine DB performance, if our DB matches to them.

In [ ]:
# ====================================
# Identify WR Routes That We Have Route Data For
# ====================================

wr_player_plays = (
    before_pass_df
    .filter(pl.col("player_position") == "WR")
)

# ------------------------------------
# Only Keep Plays That Last > 1.2 Seconds
# ------------------------------------

play_length_df = (
    wr_player_plays
    .group_by("game_play_id")
    .agg(play_length_secs = pl.max("frame_id") / 10)
)

wr_player_plays = (
    wr_player_plays
    .join(
        play_length_df, 
        on = "game_play_id", 
        how = "inner"
    )
)

wr_player_plays = (
    wr_player_plays
    .filter(
        pl.col("play_length_secs") >= 1.2
    )
)

# ----------------------------------
# Only Keep Plays with 20-100 Yards To Go To Endzone
# ----------------------------------

wr_player_plays = (
    wr_player_plays
    .filter(
        pl.col("absolute_yardline_number").is_between(
            lower_bound = 10, 
            upper_bound = 90
        )
    )
)

# ----------------------------------
# Load BDB2027 Route Data
# ----------------------------------

bdb27_route_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["player_play"]
    )
    # -------------------------
    # Only Keep Route Data
    # -------------------------
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "target", 
        "route_ran", 
    )
)

# ----------------------------------
# Load BDB2026 Route Data
# ----------------------------------

bdb26_route_df = (
    pl.read_csv(
        str(BDB_2026_SUPPLEMENTARY_DIR / "supplementary_data.csv"), 
        # Change "NA" to None
        null_values = {
            "play_action": "NA", 
            "dropback_distance": "NA"
        }
    )
    .select(
        # ID Variables
        "game_id", 
        "play_id", 
        # Helpful Variables
        "route_of_targeted_receiver", 
        "team_coverage_man_zone", 
    )
)

# --------------------------------
# Join Route Data Onto WR Player-Plays
# --------------------------------

wr_player_plays = (
    wr_player_plays
    # ---------------------------
    # Add BDB2027 Route Data
    # ---------------------------
    .join(
        bdb27_route_df, 
        on = ["game_id", "play_id", "nfl_id"], 
        how = "left"
    )
    # We will assume that if target is missing, they weren't targeted
    .with_columns(
        target = pl.col("target").fill_null(False)
    )
    # ---------------------------
    # Add BDB2026 Route Data
    # ---------------------------
    .join(
        bdb26_route_df, 
        on = ["game_id", "play_id"], 
        how = "left"
    )
    # ---------------------------
    # Create Route Variable
    # ---------------------------
    .with_columns(
        route = (
            # If target and has route_of_targeted_receiver data, then use that
            pl.when((pl.col("target") == True) & (pl.col("route_of_targeted_receiver").is_not_null()))
              .then(pl.col("route_of_targeted_receiver"))
            # Otherwise, use route_ran
              .otherwise(pl.col("route_ran"))
        )
    )
    # ---------------------------
    # Only Keep Player-Plays Where We Know The Route
    # ---------------------------
    .filter(
        pl.col("route").is_not_null()
    )
)

# ----------------------------------
# Only Keep Needed Variables
# ----------------------------------

wr_player_plays = (
    wr_player_plays
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "frame_id", 
        "game_play_id", 
        "player_play_id", 
        "absolute_yardline_number", 
        "player_name", 
        "player_position", 
        "target",
        "route", 
        "team_coverage_man_zone"
    )
)

print("Eligible WR Player-Plays:")
print("    - These routes can be covered by our DBs of interest")
print(f"    - Number of Eligible WR Player-Plays: {wr_player_plays.select("player_play_id").n_unique()}")
print(f"    - Number of Players In Eligible Player-Plays: {wr_player_plays.select("nfl_id").n_unique()}")
wr_player_plays

<br><br/>
## Pair WR To DB
***

In each route that one of our WRs ran, there should be a DB whose job is to guard that WR for the play. Our tracking data includes all DBs (CB, FS, SS) who were on the field during each player-play for our WRs of interest. We want to figure out which of those DBs were guarding each WR.

Assigning each DB to a WR is based on the following criteria: 
* Initial lateral alignment (same y value at snap)
* How close the defender is to the WR at the start of the play
* Initial downfield alignment
* The percentage of frames in which the defender is the closest defender to the WR
* Average distance throughout the play
* Average distance during the second half of the play
* Body orientation toward the WR
* Movement similarity

According to those criteria, a high assignment socre means the defender should:
* Start the play at a similar lateral position
* Start relatively close to the WR
* Have a favorable initial downfield alignment relative to the WR
* Frequently be the closest defender to the WR during the play
* Maintain a relatively short distance to the WR on average
* Stay especially close to the WR during the second half of the play
* Frequently orient their body towards the WR
* Move in a direction similar to the WR

In [ ]:
# =============================
# Specify Tracking Data
# =============================
tracking_df = before_pass_df

In [ ]:
# ==============================================================================
# CONFIGURATION & CONSTANTS
# ==============================================================================
DEFENDER_POSITIONS = ["CB", "FS", "SS"]
FACING_ANGLE_THRESHOLD = 60.0
Y_DISTANCE_SCALE = 3.0
DISTANCE_SCALE = 8.0
SECONDS_PER_FRAME = 0.1

# ==============================================================================
# STEP 1: ISOLATE WR AND DEFENDER TRACKING DATA
# ==============================================================================

# Extract Wide Receivers
wr_df = (
    tracking_df
    .filter(pl.col("player_position") == "WR")
    .select([
        "game_play_id", "player_play_id", "frame_id",
        pl.col("nfl_id").alias("wr_nfl_id"),
        pl.col("player_name").alias("wr_name"),
        "play_direction",
        pl.col("x").alias("wr_x"),
        pl.col("y").alias("wr_y"),
        pl.col("dir").alias("wr_dir"),
    ])
)

# Extract DB Defenders
defender_df = (
    tracking_df
    .filter(pl.col("player_position").is_in(DEFENDER_POSITIONS))
    .select([
        "game_play_id", "frame_id",
        pl.col("nfl_id").alias("defender_nfl_id"),
        pl.col("player_name").alias("defender_name"),
        pl.col("player_position").alias("defender_position"),
        pl.col("x").alias("defender_x"),
        pl.col("y").alias("defender_y"),
        pl.col("dir").alias("defender_dir"),
        pl.col("o").alias("defender_o"),
    ])
)

# ==============================================================================
# STEP 2: FRAME-LEVEL RELATIONSHIPS & MIDPOINT TRACKING
# ==============================================================================

play_bounds_df = (
    tracking_df
    .group_by("game_play_id")
    .agg([
        pl.col("frame_id").min().alias("min_frame"),
        pl.col("frame_id").max().alias("max_frame")
    ])
    .with_columns(
        ((pl.col("min_frame") + pl.col("max_frame")) / 2).round(0).cast(pl.Int64).alias("midpoint_frame")
    )
)

wr_defender_frames_df = (
    wr_df
    .join(defender_df, on=["game_play_id", "frame_id"], how="inner")
    .join(play_bounds_df, on="game_play_id", how="left")
    .with_columns([
        (((pl.col("wr_x") - pl.col("defender_x")) ** 2 + 
          (pl.col("wr_y") - pl.col("defender_y")) ** 2).sqrt()).alias("distance"),
          
        (pl.col("wr_y") - pl.col("defender_y")).abs().alias("y_diff"),
        
        ((pl.col("defender_x") - pl.col("wr_x")) * 
         pl.when(pl.col("play_direction") == "right").then(1.0)
         .when(pl.col("play_direction") == "left").then(-1.0)
         .otherwise(None)).alias("ahead_distance"),
         
        (pl.col("frame_id") >= pl.col("midpoint_frame")).alias("is_latter_half")
    ])
)

# ==============================================================================
# STEP 3: ANGULAR COMPONENT DERIVATIONS & NEAREST SEPARATION FLAG
# ==============================================================================

wr_defender_frames_df = (
    wr_defender_frames_df
    .with_columns(
        (
            pl.arctan2(
                pl.col("wr_x") - pl.col("defender_x"),
                pl.col("wr_y") - pl.col("defender_y")
            )
            * (180.0 / 3.141592653589793)
            + 360.0
        )
        .mod(360.0)
        .alias("bearing_defender_to_wr")
    )
    .with_columns(
        (
            ((pl.col("defender_o") - pl.col("bearing_defender_to_wr") + 180.0).mod(360.0) + 360.0).mod(360.0) - 180.0
        )
        .abs()
        .alias("facing_angle_diff"),
        
        (
            ((pl.col("defender_dir") - pl.col("wr_dir") + 180.0).mod(360.0) + 360.0).mod(360.0) - 180.0
        )
        .alias("movement_angle_diff")
    )
    .with_columns(
        (pl.col("facing_angle_diff") <= FACING_ANGLE_THRESHOLD).alias("facing_wr"),
        (
            (pl.col("movement_angle_diff") * (3.141592653589793 / 180.0))
            .cos()
            .abs()
        )
        .alias("movement_similarity")
    )
)

nearest_by_frame_df = (
    wr_defender_frames_df
    .sort(["player_play_id", "frame_id", "distance"])
    .group_by(["player_play_id", "frame_id"], maintain_order=True)
    .first()
    .select([
        "player_play_id", "frame_id", 
        pl.col("defender_nfl_id").alias("nearest_defender_nfl_id")
    ])
)

wr_defender_frames_df = (
    wr_defender_frames_df
    .join(nearest_by_frame_df, on=["player_play_id", "frame_id"], how="left")
    .with_columns(
        (pl.col("defender_nfl_id") == pl.col("nearest_defender_nfl_id")).alias("is_nearest")
    )
)

# ==============================================================================
# STEP 4: AGGREGATE PLAY-LEVEL SUMMARY METRICS
# ==============================================================================

candidate_summary_df = (
    wr_defender_frames_df
    .group_by([
        "game_play_id", "player_play_id", "wr_nfl_id", "wr_name",
        "defender_nfl_id", "defender_name", "defender_position"
    ])
    .agg([
        pl.len().alias("n_observed_frames"),
        pl.col("is_nearest").mean().alias("nearest_frame_pct"),
        pl.col("distance").mean().alias("mean_distance"),
        pl.col("distance").filter(pl.col("is_latter_half")).mean().alias("latter_half_mean_distance"),
        pl.col("y_diff").mean().alias("mean_y_diff"),
        pl.col("facing_wr").cast(pl.Float64).mean().alias("facing_wr_pct"),
        pl.col("movement_similarity").mean().alias("movement_similarity")
    ])
)

initial_alignment_df = (
    wr_defender_frames_df
    .filter(pl.col("frame_id") == 1)
    .select([
        "game_play_id", "player_play_id", "defender_nfl_id",
        pl.col("distance").alias("initial_distance"),
        pl.col("y_diff").alias("initial_y_diff"),
        pl.col("ahead_distance").alias("initial_ahead_distance")
    ])
)

# Clean join explicitly including game_play_id mapping
candidate_summary_df = candidate_summary_df.join(
    initial_alignment_df, 
    on=["game_play_id", "player_play_id", "defender_nfl_id"], 
    how="left"
)

# ==============================================================================
# STEP 5: CONFIDENCE SCORING & DUAL-LOCK 1-ON-1 ASSIGNMENTS
# ==============================================================================

# Compile exploratory metrics, handling potential edge case nulls from snap framing gracefully
candidate_summary_df = (
    candidate_summary_df
    .with_columns([
        (-pl.col("initial_y_diff").fill_null(99.0) / Y_DISTANCE_SCALE).exp().alias("initial_y_score"),
        (-pl.col("initial_distance").fill_null(99.0) / DISTANCE_SCALE).exp().alias("initial_distance_score"),
        ((pl.col("initial_ahead_distance").fill_null(-2.0) + 2.0) / 6.0).clip(0.0, 1.0).alias("initial_ahead_score"),
        (-pl.col("mean_distance") / DISTANCE_SCALE).exp().alias("mean_distance_score"),
        (-pl.col("latter_half_mean_distance").fill_null(pl.col("mean_distance")) / DISTANCE_SCALE).exp().alias("latter_half_distance_score")
    ])
    .with_columns([
        (
            0.05 * pl.col("initial_y_score") +
            0.05 * pl.col("initial_ahead_score") +
            0.05 * pl.col("initial_distance_score") +
            0.25 * pl.col("nearest_frame_pct") +
            0.15 * pl.col("mean_distance_score") +
            0.30 * pl.col("latter_half_distance_score") + # Heavy focus on downfield route tracking
            0.08 * pl.col("facing_wr_pct") +
            0.07 * pl.col("movement_similarity")
        ).alias("assignment_score")
    ])
)

# Enforce strict 1-on-1 mutual exclusivity across the play
assignments_df = (
    candidate_summary_df
    .filter(pl.col("assignment_score") >= 0.80)
    .sort("assignment_score", descending=True)
    # Lock 1: Ensure a single DB isn't assigned to multiple wideouts
    .with_columns(
        pl.col("assignment_score").rank(method="ordinal", descending=True)
        .over(["game_play_id", "defender_nfl_id"]).alias("defender_lock_rank")
    )
    # Lock 2: Ensure a single wideout isn't claiming multiple DB trackers
    .with_columns(
        pl.col("assignment_score").rank(method="ordinal", descending=True)
        .over(["player_play_id"]).alias("wr_lock_rank")
    )
    # Filter for valid unique pairs
    .filter((pl.col("defender_lock_rank") == 1) & (pl.col("wr_lock_rank") == 1))
    # Select exact 8 columns matching the target assignment schema requirement
    .select([
        "game_play_id",
        "player_play_id",
        "wr_nfl_id",
        "wr_name",
        pl.col("defender_nfl_id").alias("assigned_defender_nfl_id"),
        pl.col("defender_name").alias("assigned_defender_name"),
        pl.col("defender_position").alias("assigned_defender_position"),
        "assignment_score"
    ])
    .sort(["game_play_id", "player_play_id"])
)

# ==============================================================================
# STEP 6: BUILD GENERATED FRAME-LEVEL SEPARATION PROFILE
# ==============================================================================

# Re-extract clean sequential tracking vectors to blend into the complete metrics dataframe
db_separation_df = (
    wr_df
    # Align the assignment maps to their matching tracking records
    .join(assignments_df, on=["game_play_id", "player_play_id", "wr_nfl_id", "wr_name"], how="inner")
    .with_columns(
        pl.col("wr_nfl_id").alias("wr_nfl_id_right")
    )
    # Join the matching frame-by-frame coordinate updates from the assigned defensive back
    .join(
        defender_df,
        left_on=["game_play_id", "frame_id", "assigned_defender_nfl_id"],
        right_on=["game_play_id", "frame_id", "defender_nfl_id"],
        how="inner"
    )
    .with_columns([
        pl.col("defender_name"),
        pl.col("defender_position"),
        (((pl.col("wr_x") - pl.col("defender_x")) ** 2 + 
          (pl.col("wr_y") - pl.col("defender_y")) ** 2).sqrt()).alias("wr_cb_distance")
    ])
)

In [ ]:
# ======================================
# Review Assignments, See If It Looks Accurate
# ======================================

# --------------------------------------
# Assignments Must Have This Assignment Score
# --------------------------------------

MIN_ASSIGNMENT_SCORE = 0.75

assignments_df = (
    assignments_df
    .filter(pl.col("assignment_score") >= MIN_ASSIGNMENT_SCORE)
)

assignments_df

In [ ]:
# ======================================
# Tracking Data With Pairings
# ======================================

# --------------------------------------
# Assignments Must Have This Assignment Score
# --------------------------------------

MIN_ASSIGNMENT_SCORE = 0.75

db_separation_df = (
    db_separation_df
    .filter(pl.col("assignment_score") >= MIN_ASSIGNMENT_SCORE)
)

db_separation_df

In [ ]:
# =======================================
# Clean Up assignments_df
# =======================================

assignments_df = (
    assignments_df
    # ------------------------------
    # Add game_id, play_id
    # ------------------------------
    .join(
        player_play_df
        .select(
            "game_play_id", 
            "game_id", 
            "play_id"
        )
        .unique(),
        on = "game_play_id", 
        how = "inner"
    )
    .rename({
        "assigned_defender_nfl_id": "defender_nfl_id", 
        "assigned_defender_name": "defender_name", 
        "assigned_defender_position": "defender_position", 
        "wr_nfl_id": "assigned_wr_nfl_id",
        "wr_name": "assigned_wr_name"
    })
    .with_columns(
        player_play_id = pl.col("game_play_id") + "_" + pl.col("defender_nfl_id").cast(pl.String)
    )
    .with_columns(
        wr_player_play_id = pl.col("game_play_id") + "_" + pl.col("assigned_wr_nfl_id").cast(pl.String)
    )
    .select(
        "game_id", 
        "play_id", 
        "game_play_id", 
        "player_play_id", 
        "assignment_score", 
        "defender_nfl_id", 
        "defender_name",
        "defender_position", 
        "assigned_wr_nfl_id", 
        "assigned_wr_name",
        "wr_player_play_id"
    )
)

# ----------------------------------
# Only Keep DB Player-Plays If The DB Was Guarding
# An Eligible WR Player-Play
# ----------------------------------

assignments_df = (
    assignments_df
    .join(
        wr_player_plays.drop("frame_id").unique(), 
        left_on = ["game_id", "play_id", "game_play_id", "assigned_wr_nfl_id", "wr_player_play_id"],
        right_on = ["game_id", "play_id", "game_play_id", "nfl_id", "player_play_id"], 
        how = "inner"
    )
)

print("DB/WR Assignments:")
print(f"    - Number of DB Player-Plays: {assignments_df.select("player_play_id").n_unique()}")
print(f"    - Number of DBs: {assignments_df.select("defender_nfl_id").n_unique()}")
print(f"    - Number of Unique Plays: {assignments_df.select("game_play_id").n_unique()}")
assignments_df.filter(pl.col("team_coverage_man_zone") == "ZONE_COVERAGE", pl.col("defender_position").is_in(["FS", "SS"]))

In [ ]:
# ======================================
# Clean Up db_separation_df
# ======================================

db_separation_df = (
    db_separation_df
    # ------------------------------
    # Add game_id, play_id
    # ------------------------------
    .join(
        player_play_df
        .select(
            "game_play_id", 
            "game_id", 
            "play_id"
        )
        .unique(),
        on = "game_play_id", 
        how = "inner"
    )
    .rename({
        "assigned_defender_nfl_id": "defender_nfl_id",
        "wr_nfl_id": "assigned_wr_nfl_id", 
        "wr_name": "assigned_wr_name"
    })
    .with_columns(
        player_play_id = pl.col("game_play_id") + "_" + pl.col("defender_nfl_id").cast(pl.String)
    )
    .with_columns(
        wr_player_play_id = pl.col("game_play_id") + "_" + pl.col("assigned_wr_nfl_id").cast(pl.String)
    )
    .select(
        "game_id", 
        "play_id", 
        "game_play_id", 
        "player_play_id", 
        "frame_id", 
        "assignment_score",
        "play_direction",
        "defender_nfl_id", 
        "defender_name", 
        "defender_position",
        "defender_x", 
        "defender_y", 
        "defender_dir", 
        "defender_o", 
        "assigned_wr_nfl_id", 
        "assigned_wr_name", 
        "wr_player_play_id",
        "wr_x", 
        "wr_y", 
        "wr_dir",
        "wr_cb_distance"
    )
)

# ----------------------------------
# Only Keep DB Player-Plays If The DB Was Guarding
# An Eligible WR Player-Play
# ----------------------------------

db_separation_df = (
    db_separation_df
    .join(
        wr_player_plays, 
        left_on = ["game_id", "play_id", "game_play_id", "assigned_wr_nfl_id", "wr_player_play_id", "frame_id"],
        right_on = ["game_id", "play_id", "game_play_id", "nfl_id", "player_play_id", "frame_id"], 
        how = "inner"
    )
)

print(f"Assignments That Meet Assignment Similarity Threshold:")
print(f"    - Number of DB Player-Plays: {db_separation_df.select("player_play_id").n_unique()}")
print(f"    - Number of DBs: {db_separation_df.select("defender_nfl_id").n_unique()}")
print(f"    - Number of Unique Plays: {db_separation_df.select("game_play_id").n_unique()}")
db_separation_df

In [ ]:
# ======================================
# Play Exposure
# ======================================

print("Defenders With >= 10 Plays:")
(
    db_separation_df
    .group_by("defender_nfl_id")
    .agg(num_plays = pl.col("player_play_id").n_unique())
    .filter(pl.col("num_plays") >= 10)
    .sort("num_plays", descending = True)
)


<br><br/>
## Export Data
***

In [ ]:
# =================================================

# =================================================
# Player-Time Level WR/CB Separtion Data
# =================================================

# This dataset has 1 row per player-time from a player-play from 
# one of our WRs of interest. It includes player-tracking data for
# the player and their assigned defender. It also includes their
# separation (distance between WR and DB) at every player-time.

db_separation_df.write_csv(str(SEPARATION_DATA_DIR / "11_2023_Rookie_DB_Player_Time_Level_Separation_Data.csv"))